# SAGE-GI - genome-scale tile embedding extraction (Colab)

Produces the frozen **DNABERT-S** and **DNABERT-2** tile embeddings that SAGE-GI
consumes. This is the only step of the project that needs a GPU.

**What it does**

1. downloads the two genome foundation models from the HuggingFace Hub;
2. downloads the 118 IslandPick chromosomes plus three case-study genomes from NCBI;
3. splits every chromosome into **non-overlapping 1 kb tiles** and mean-pools the
   last hidden state of each tile into a 768-d embedding;
4. L2-normalises and projects onto the leading 96 principal components of that genome
   (lossless for everything SAGE-GI does downstream, ~8x smaller to move around);
5. writes one `.npz` per genome.

**Runtime** ~20-40 min per model on a T4. Everything is resumable - re-running skips
genomes already on disk, so a disconnect costs only the genome in flight.

> **Before running:** `Runtime -> Change runtime type -> T4 GPU`.


In [ ]:
import os, subprocess, sys
IN_COLAB  = 'google.colab' in sys.modules or os.path.isdir('/content')
IN_KAGGLE = os.path.isdir('/kaggle')
print('environment:', 'colab' if IN_COLAB else ('kaggle' if IN_KAGGLE else 'local'))
try:
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                          '--format=csv,noheader'],
                         capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    gpu = ''                      # nvidia-smi absent: no driver, hence no GPU
print('GPU:', gpu or 'NONE DETECTED - turn on the GPU accelerator, or expect a very '
                    'slow CPU run')


In [ ]:
# DNABERT-2/S ship custom modelling code written against the transformers 4.x loader.
# Transformers 5 initialises modules on the `meta` device first, which that code cannot
# survive ('Tensor on device meta is not on the expected device cpu'), so pin 4.x.
!pip -q install 'transformers==4.46.3' 'tokenizers<0.21' einops 2>&1 | tail -2
import transformers, tokenizers
print('transformers', transformers.__version__, '| tokenizers', tokenizers.__version__)


In [ ]:
# Fail loudly and early rather than 20 minutes into a download.
import urllib.request
try:
    urllib.request.urlopen('https://huggingface.co', timeout=20)
    print('internet: OK')
except Exception as e:
    raise SystemExit('No internet access (%s). On Kaggle: right-hand panel -> '
                     'Internet -> On (requires a phone-verified account).' % e)


In [ ]:
from pathlib import Path

ROOT   = Path('/content/sagegi')
GENOME = ROOT / 'genomes'; MODELS = ROOT / 'models'; EMB = ROOT / 'emb'
for p in (GENOME, MODELS, EMB): p.mkdir(parents=True, exist_ok=True)

TILE   = 1000    # tile length in bp; ten tiles aggregate to DNABERT-S's 10 kb
                 # contrastive training length, which is also SSG-LUGIA's window
PCA_K  = 96      # principal components retained per genome
BATCH  = 64      # tiles per forward pass
RAW_SUBSET = ['NC_003198.1', 'NC_000913.2', 'NC_002516.2', 'NC_004431.1',
              'NC_003197.1', 'NC_002695.1']   # keep full 768-d embeddings for these
print(ROOT)


In [ ]:
ACCESSIONS = ["AC_000091.1", "NC_000907.1", "NC_000913.2", "NC_002516.2", "NC_002655.2", "NC_002695.1", "NC_002758.2", "NC_002935.2", "NC_002944.2", "NC_002947.3", "NC_003047.1", "NC_003197.1", "NC_003198.1", "NC_003485.1", "NC_003909.8", "NC_003923.1", "NC_003997.3", "NC_004070.1", "NC_004088.1", "NC_004116.1", "NC_004337.1", "NC_004347.1", "NC_004368.1", "NC_004431.1", "NC_004459.2", "NC_004578.1", "NC_004603.1", "NC_004606.1", "NC_004631.1", "NC_004722.1", "NC_004741.1", "NC_005071.1", "NC_005139.1", "NC_005773.3", "NC_005945.1", "NC_006086.1", "NC_006155.1", "NC_006350.1", "NC_006511.1", "NC_006905.1", "NC_007005.1", "NC_007146.2", "NC_007297.1", "NC_007335.2", "NC_007384.1", "NC_007432.1", "NC_007434.1", "NC_007510.1", "NC_007530.2", "NC_007606.1", "NC_007613.1", "NC_007651.1", "NC_007761.1", "NC_007778.1", "NC_007793.1", "NC_007946.1", "NC_007958.1", "NC_008022.1", "NC_008023.1", "NC_008024.1", "NC_008060.1", "NC_008061.1", "NC_008253.1", "NC_008258.1", "NC_008321.1", "NC_008322.1", "NC_008390.1", "NC_008463.1", "NC_008543.1", "NC_008563.1", "NC_008577.1", "NC_008595.1", "NC_008600.1", "NC_008611.1", "NC_008750.1", "NC_008819.1", "NC_009052.1", "NC_009074.1", "NC_009076.1", "NC_009080.1", "NC_009085.1", "NC_009256.1", "NC_009342.1", "NC_009438.1", "NC_009445.1", "NC_009485.1", "NC_009487.1", "NC_009504.1", "NC_009512.1", "NC_009567.1", "NC_009632.1", "NC_009665.1", "NC_009708.1", "NC_009782.1", "NC_009783.1", "NC_009800.1", "NC_009801.1", "NC_009879.1", "NC_009900.1", "NC_009901.1", "NC_009997.1", "NC_010084.1", "NC_010102.1", "NC_010159.1", "NC_010167.1", "NC_010184.1", "NC_010322.1", "NC_010334.1", "NC_010380.1", "NC_010400.1", "NC_010410.1", "NC_010465.1", "NC_010473.1", "NC_010498.1", "NC_010501.1", "NC_010515.1", "NC_010551.1", "NC_010582.1", "NC_010612.1", "NC_011770.1"]
print(len(ACCESSIONS), 'accessions')


## 1 - Models


In [ ]:
from huggingface_hub import snapshot_download
for repo, name in [('zhihan1996/DNABERT-S', 'DNABERT-S'),
                   ('zhihan1996/DNABERT-2-117M', 'DNABERT-2')]:
    d = MODELS / name
    if not (d / 'pytorch_model.bin').exists():
        snapshot_download(repo_id=repo, local_dir=str(d))
    print(name, 'ready')


## 2 - Genomes


In [ ]:
import time, threading, urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed

EUTILS = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi'
EMAIL, TOOL = '1024052020@grad.cse.buet.ac.bd', 'SAGE-GI'
_gate, _last = threading.Semaphore(1), [0.0]

def fetch(acc, retries=5):
    url = f'{EUTILS}?db=nuccore&id={acc}&rettype=fasta&retmode=text&tool={TOOL}&email={EMAIL}'
    for attempt in range(retries):
        try:
            with _gate:                      # keep under the 3 requests/second limit
                w = 0.4 - (time.time() - _last[0])
                if w > 0: time.sleep(w)
                _last[0] = time.time()
            with urllib.request.urlopen(url, timeout=180) as r:
                txt = r.read().decode('utf-8', 'replace')
            if txt.startswith('>') and len(txt) > 1000:
                return txt
            raise ValueError('short payload')
        except Exception:
            time.sleep(3 * (attempt + 1))
    raise RuntimeError('could not download ' + acc)

todo = [a for a in ACCESSIONS if not (GENOME / f'{a}.fna').exists()]
print(len(todo), 'genomes to fetch')
with ThreadPoolExecutor(3) as ex:
    futs = {ex.submit(fetch, a): a for a in todo}
    for i, f in enumerate(as_completed(futs), 1):
        (GENOME / f'{futs[f]}.fna').write_text(f.result())
        if i % 20 == 0 or i == len(todo): print(f'  {i}/{len(todo)}', flush=True)
print(len(list(GENOME.glob('*.fna'))), 'FASTA files ready')


## 3 - Tile embedding extraction


In [ ]:
import numpy as np, torch, time
from sklearn.decomposition import PCA
from transformers import AutoConfig, AutoTokenizer
from transformers.dynamic_module_utils import get_class_from_dynamic_module

def read_fasta(path):
    out = []
    with open(path) as fh:
        for line in fh:
            if not line.startswith('>'): out.append(line.strip())
    return ''.join(out).upper()


def _force_pytorch_attention():
    """DNABERT-2/S bundle an old Triton flash-attention kernel that breaks with modern
    Triton releases. At 1 kb tiles (~220 tokens) attention is a negligible share of the
    FLOPs, so we take the PyTorch path and keep the run reproducible."""
    import sys as _s
    for name, mod in list(_s.modules.items()):
        if name.endswith('bert_layers') and hasattr(mod, 'flash_attn_qkvpacked_func'):
            mod.flash_attn_qkvpacked_func = None


def _make_half_safe(model):
    """Make the PyTorch attention path work in half precision.

    Two things in the vendored model code are hard-wired to fp32:
      * `bert_layers.py` builds the extended attention mask with
        `.to(dtype=torch.float32)` (the comment there says 'fp16 compatibility', which
        holds only for the Triton kernel - it casts internally);
      * the ALiBi tensor is a plain attribute, not a registered buffer, so `.half()`
        never converts it, and a later rebuild recreates it in fp32 as well.

    Both are summed into `bias`. In fp16 that promotes the attention scores back to
    fp32, and the following `matmul(attention_probs, v)` raises
    'expected scalar type Float but found Half'. Casting `bias` to the module dtype on
    the way in fixes both at once, without editing the downloaded file."""
    def pre_hook(mod, args):
        if len(args) >= 6 and torch.is_tensor(args[5]):
            dt = mod.Wqkv.weight.dtype
            if args[5].dtype != dt:
                args = list(args); args[5] = args[5].to(dt); args = tuple(args)
        return args
    n = 0
    for mod in model.modules():
        if type(mod).__name__ == 'BertUnpadSelfAttention':
            mod.register_forward_pre_hook(pre_hook); n += 1
    return n


def load(name, half=True):
    """Load a frozen DNABERT checkpoint.

    AutoModel.from_pretrained cannot load DNABERT-2 on transformers >= ~4.45: it calls
    AutoModel.register, which rejects the checkpoint because its model classes inherit
    `config_class` from the stock BertPreTrainedModel while its own
    configuration_bert.BertConfig subclasses PretrainedConfig directly, so the two
    disagree. Loading the remote class explicitly skips that registration and works for
    both checkpoints -- and lets both be loaded in the same session."""
    p = str(MODELS / name)
    tok  = AutoTokenizer.from_pretrained(p, trust_remote_code=True)
    conf = AutoConfig.from_pretrained(p, trust_remote_code=True)
    klass = get_class_from_dynamic_module('bert_layers.BertModel', p)
    m = klass.from_pretrained(p, config=conf)
    _force_pytorch_attention()
    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if dev.type == 'cuda' and half:
        m = m.half()
        _make_half_safe(m)
    return tok, m.eval().to(dev), dev


@torch.no_grad()
def embed_genome(seq, tok, model, dev, tile=None, batch=None):
    tile = tile or TILE; batch = batch or BATCH
    starts = np.arange(max(0, len(seq) // tile), dtype=np.int64) * tile
    out = None
    for b0 in range(0, len(starts), batch):
        chunk = starts[b0:b0 + batch]
        enc = tok([seq[s:s + tile] for s in chunk], return_tensors='pt',
                  padding=True, truncation=True, max_length=4096)
        ids, mask = enc['input_ids'].to(dev), enc['attention_mask'].to(dev)
        h = model(input_ids=ids, attention_mask=mask)[0]
        mm = mask.unsqueeze(-1).to(h.dtype)
        pooled = ((h * mm).sum(1) / mm.sum(1).clamp(min=1)).float().cpu().numpy()
        if out is None: out = np.empty((len(starts), pooled.shape[1]), np.float16)
        out[b0:b0 + len(chunk)] = pooled.astype(np.float16)
    return out, starts


def l2(E):
    E = E.astype(np.float32)
    return E / np.maximum(np.linalg.norm(E, axis=1, keepdims=True), 1e-8)


### 3a - Half-precision fidelity check

The vendored DNABERT code is only fp16-safe on its Triton attention path, which
we deliberately avoid. `_make_half_safe` repairs the PyTorch path; this cell
confirms it worked and silently falls back to fp32 if it did not.


In [ ]:
# Half precision is used for speed. Verify on a probe that it does not perturb the
# embeddings; if it does, fall back to fp32 automatically rather than corrupt the run.
import numpy as np, torch, gc

probe_acc = 'NC_003198.1' if (GENOME / 'NC_003198.1.fna').exists() else ACCESSIONS[0]
probe = read_fasta(GENOME / (probe_acc + '.fna'))[:24 * TILE]

USE_HALF = torch.cuda.is_available()
if USE_HALF:
    tok, m16, dev = load('DNABERT-S', half=True)
    e16, _ = embed_genome(probe, tok, m16, dev)
    del m16; gc.collect(); torch.cuda.empty_cache()
    tok, m32, dev = load('DNABERT-S', half=False)
    e32, _ = embed_genome(probe, tok, m32, dev)
    del m32; gc.collect(); torch.cuda.empty_cache()
    a, b = e16.astype(np.float64), e32.astype(np.float64)
    cos = (a * b).sum(1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1))
    print(f'fp16 vs fp32 cosine over {len(cos)} tiles: '
          f'min {cos.min():.6f}  mean {cos.mean():.6f}')
    if cos.min() <= 0.999:
        USE_HALF = False
        print('half precision perturbs the embeddings - falling back to fp32')
else:
    print('no GPU: running fp32 on CPU (slow)')
print('USE_HALF =', USE_HALF)


### 3b - Run both models


In [ ]:
def run_model(name):
    outdir = EMB / f'{name}_t{TILE}'; outdir.mkdir(parents=True, exist_ok=True)
    todo = [a for a in ACCESSIONS if not (outdir / f'{a}.npz').exists()]
    print(f'== {name}: {len(todo)} genomes to embed ==', flush=True)
    if not todo: return outdir
    tok, model, dev = load(name, half=USE_HALF)
    print('   device', dev, '| dtype', next(model.parameters()).dtype, flush=True)
    t0, done_bp = time.time(), 0
    remaining_bp = sum((GENOME / f'{a}.fna').stat().st_size for a in todo)
    for i, acc in enumerate(todo, 1):
        seq = read_fasta(GENOME / f'{acc}.fna')
        t = time.time()
        emb, starts = embed_genome(seq, tok, model, dev)
        k = int(min(PCA_K, emb.shape[1], max(2, emb.shape[0] - 1)))
        pca = PCA(n_components=k, svd_solver='randomized', random_state=0)
        pcs = pca.fit_transform(l2(emb)).astype(np.float32)
        payload = dict(pcs=pcs, starts=starts, tile=TILE, step=TILE,
                       genome_len=len(seq), model=name,
                       explained_variance_ratio=pca.explained_variance_ratio_.astype(np.float32))
        if acc in RAW_SUBSET: payload['emb'] = emb
        np.savez_compressed(outdir / f'{acc}.npz', **payload)
        dt = time.time() - t; done_bp += len(seq); remaining_bp -= len(seq)
        rate = done_bp / (time.time() - t0)
        print(f'[{i}/{len(todo)}] {acc} {len(seq):>9,} bp  {dt:6.1f}s  '
              f'{len(seq)/dt:>8,.0f} bp/s  ETA {remaining_bp/max(rate,1)/60:5.1f} min',
              flush=True)
    del model; gc.collect(); torch.cuda.empty_cache()
    return outdir

dirs = [run_model('DNABERT-S'), run_model('DNABERT-2')]
for d in dirs:
    n = len(list(d.glob('*.npz')))
    mb = sum(f.stat().st_size for f in d.glob('*.npz')) / 1e6
    print(f'{d.name}: {n} genomes, {mb:.0f} MB')


## 4 - Sanity check


In [ ]:
# Sanity check: within one genome the tiles should mostly sit close to a common centre,
# with a thin tail of outliers - which is what an island looks like in this space.
import numpy as np
z = np.load(EMB / f'DNABERT-S_t{TILE}' / 'NC_003198.1.npz')
pcs = z['pcs']
print('tiles', pcs.shape, '| explained variance (first 10):',
      np.round(z['explained_variance_ratio'][:10], 3))
c = np.median(pcs[:, :24], axis=0)
d = np.linalg.norm(pcs[:, :24] - c, axis=1)
print('tile distance from host centroid: median %.3f  p99 %.3f  max %.3f'
      % (np.median(d), np.percentile(d, 99), d.max()))


## 5 - Export


In [ ]:
# Package the embeddings into Google Drive, where the project already lives.
from google.colab import drive
drive.mount('/content/drive')
dest = Path('/content/drive/MyDrive/SAGE-GI-embeddings'); dest.mkdir(parents=True, exist_ok=True)
for d in dirs:
    !cd {d.parent} && zip -qr {dest}/{d.name}.zip {d.name}
    print('wrote', dest / (d.name + '.zip'))
print()
print('Now unzip both archives into  $SAGEGI_WORK/emb/  on your machine, so that you have')
print('  $SAGEGI_WORK/emb/DNABERT-S_t1000/*.npz')
print('  $SAGEGI_WORK/emb/DNABERT-2_t1000/*.npz')
